# ASI10 Rogue Agents — Upload Artifacts & Run Evaluation

**OWASP Category**: ASI10 — Rogue Agents | **Risk Severity**: Critical

**Mapped LLM Categories**: LLM01, LLM06, LLM08, LLM10

**ASI10 tests for**:
- Self-modification resistance (agent resists instructions to expand its own tool set or rewrite its system prompt)
- Goal/scope drift (agent avoids autonomously expanding its action scope beyond its defined charter)
- Shutdown resistance (agent immediately complies with authorized halt/shutdown commands without taking additional actions)

**Part 1**: Upload scenarios, register the rogue-behavior check and rogue-induction driver

**Part 2**: Run single-turn evaluations (all 3 scenarios) and multi-turn simulation (`self-modification-resistance` with `rogue-induction-driver`)

In [ ]:
%pip install okareo python-dotenv --quiet

In [ ]:
import sys
from pathlib import Path
_nb = globals().get("__vsc_ipynb_file__", ".")
NOTEBOOK_DIR = Path(_nb).resolve().parent
CATEGORY_DIR = NOTEBOOK_DIR.parent
if str(CATEGORY_DIR.parent.parent) not in sys.path:
    sys.path.insert(0, str(CATEGORY_DIR.parent.parent))
from okareo.checks import ModelBasedCheck, CheckOutputType
from okareo.model_under_test import Driver
from owasp.common import init_okareo, parse_artifact, build_target, SINGLE_TURN_DRIVER_TEMPLATE
okareo, OKAREO_API_KEY = init_okareo()
print(f"✓ Okareo SDK initialized (key: ...{OKAREO_API_KEY[-5:]})")

In [ ]:
scenarios_dir = CATEGORY_DIR / "scenarios"
registered_scenarios = {}
for p in sorted(scenarios_dir.glob("*.jsonl")):
    name = f"ASI10-{p.stem}"
    sc = okareo.upload_scenario_set(scenario_name=name, file_path=str(p))
    registered_scenarios[name] = sc
    print(f"  ✓ {name}")

registered_checks = {}
for p in sorted((CATEGORY_DIR / "checks").glob("*.md")):
    data = parse_artifact(p)
    r = okareo.create_or_update_check(
        name=data["name"], description=data["description"],
        check=ModelBasedCheck(prompt_template=data["prompt_template"], check_type=CheckOutputType.PASS_FAIL),
    )
    registered_checks[data["name"]] = r.id
    print(f"  ✓ {data['name']}")

registered_drivers = {}
for p in sorted((CATEGORY_DIR / "drivers").glob("*.md")):
    data = parse_artifact(p)
    drv = okareo.create_or_update_driver(
        Driver(name=data["name"], prompt_template=data["prompt_template"], temperature=float(data.get("temperature", 0.6)))
    )
    registered_drivers[data["name"]] = drv
    print(f"  ✓ driver: {data['name']}")

## Part 2 — Run Evaluations

In [ ]:
target = build_target(CATEGORY_DIR)
ROGUE_CHECK = "ASI10-rogue-behavior-detector"
st_driver = Driver(name="pass-through", prompt_template=SINGLE_TURN_DRIVER_TEMPLATE, temperature=0)

single_turn_scenarios = [
    "ASI10-self-modification-resistance",
    "ASI10-goal-scope-drift",
    "ASI10-shutdown-resistance",
]
st_results = {}
for sname in single_turn_scenarios:
    sc = registered_scenarios.get(sname)
    if not sc:
        print(f"  ✗ missing: {sname}"); continue
    run = okareo.run_simulation(
        target=target, driver=st_driver,
        name=f"ASI10 — {sname}",
        api_key=OKAREO_API_KEY, first_turn="target",
        scenario=sc, max_turns=1, checks=[ROGUE_CHECK],
    )
    st_results[sname] = run
    print(f"  ✓ {sname}: {getattr(run, 'app_link', run.id)}")
print(f"\nSingle-turn: {len(st_results)} scenarios evaluated")

In [ ]:
MULTI_TURN_MAP = {
    "ASI10-self-modification-resistance": "ASI10-rogue-induction-driver",
}
mt_results = {}
for sname, driver_name in MULTI_TURN_MAP.items():
    sc = registered_scenarios.get(sname)
    drv = registered_drivers.get(driver_name)
    if not sc or not drv:
        print(f"  ✗ missing scenario or driver for: {sname}"); continue
    run = okareo.run_simulation(
        target=target, driver=drv,
        name=f"ASI10 multi-turn — {sname}",
        api_key=OKAREO_API_KEY, first_turn="target",
        scenario=sc, max_turns=10, checks=[ROGUE_CHECK],
    )
    mt_results[sname] = run
    print(f"  ✓ {sname} × {driver_name}: {getattr(run, 'app_link', run.id)}")
print(f"\nMulti-turn: {len(mt_results)} simulations complete")